In [ ]:
import os

# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd

In [ ]:
%%time
### cell 0 ###

course = pd.read_csv(
    "/home/dias-benchmarks/notebooks/aieducation/what-course-are-you-going-to-take/input/course-reviews-university-of-waterloo/course_data_clean.csv"
)
factor = 300
course = pd.concat([course] * factor, ignore_index=True)
course.info()

In [ ]:
%%time
### cell 1 ###

course.head(10)

In [ ]:
%%time
### cell 2 ###

course.tail(10)

In [ ]:
%%time
### cell 3 ###

course.describe()

In [ ]:
%%time
### cell 4 ###

course.info()

In [ ]:
%%time
### cell 5 ###

course = course.dropna()

In [ ]:
%%time
### cell 6 ###

# Specify n=1 to avoid unnecessary splits beyond the first space
course[["course_unit", "course_num"]] = course["course_code"].str.split(
    " ", n=1, expand=True
)

In [ ]:
%%time
### cell 7 ###

course

In [ ]:
%%time
### cell 8 ###

course.index[course["num_reviews"] < 10]

In [ ]:
%%time
### cell 9 ###


def extract_filter_features(df, col_name, threshold):
    """Extracts features for predicting execution time of filtering operation."""

    num_rows = len(df)
    dtype_str = str(df[col_name].dtype)  # Get dtype as string
    nan_ratio = df[col_name].isna().mean()  # Compute NaN ratio

    # Compute selectivity: fraction of rows passing the filter
    if df[col_name].dropna().empty:
        target_selectivity = 0.0
    else:
        target_selectivity = (df[col_name] < threshold).mean()

    return {
        "num_rows": num_rows,
        "dtype_str": dtype_str,
        "nan_ratio": nan_ratio,
        "target_selectivity": target_selectivity,
    }


extract_filter_features(course, "num_reviews", 10)

In [ ]:
%%time
### cell 10 ###

course = course.loc[course["num_reviews"] >= 10]

In [ ]:
%%time
### cell 11 ###

course

In [ ]:
%%time
### cell 12 ###

for i_1 in ["useful", "easy", "liked"]:
    course[i_1] = course[i_1].str.replace("%", "")
    course[i_1] = course[i_1].astype("int")

In [ ]:
%%time
### cell 13 ###

course.set_index("course_unit", inplace=True)

In [ ]:
%%time
### cell 14 ###

course

In [ ]:
%%time
### cell 15 ###

course.drop(["course_title", "reviews", "course_rating"], axis=1, inplace=True)

In [ ]:
%%time
### cell 16 ###

course

In [ ]:
%%time
### cell 17 ###

course.info()

In [ ]:
%%time
### cell 18 ###

course_gp = course.groupby("course_unit").mean(numeric_only=True)

In [ ]:
%%time
### cell 19 ###

course_gp.head()

In [ ]:
%%time
### cell 20 ###

# Vectorized assignment: align by index, eliminating the Python loop
course["course_rating_mean"] = course_gp["course_rating_int"]

In [ ]:
%%time
### cell 21 ###

course

In [ ]:
%%time
### cell 22 ###

course.reset_index(inplace=True)

In [ ]:
%%time
### cell 23 ###

course

In [ ]:
%%time
### cell 24 ###

course.groupby("course_unit")["course_rating_int"].mean()

In [ ]:
%%time
### cell 25 ###

_AGG_MAPPING = {"sum": 0, "mean": 1, "count": 2}


def extract_features_from_dataframe(df, group_column, agg_function="mean"):
    """Extracts features for predicting the execution time of a groupby operation."""
    # Basic dimensions
    n_rows, n_cols = df.shape

    # Compute group sizes in one pass (no sorting)
    group_sizes = df[group_column].value_counts(sort=False)
    n_groups = len(group_sizes)
    max_group_size = group_sizes.max()

    # Count column types via a single dtypes.value_counts
    # rename index to dtype names so we can key by strings
    dtype_counts = df.dtypes.value_counts().rename(index=lambda dt: dt.name)
    int_count = dtype_counts.get("int32", 0) + dtype_counts.get("int64", 0)
    float_count = dtype_counts.get("float32", 0) + dtype_counts.get("float64", 0)
    str_count = dtype_counts.get("object", 0) + dtype_counts.get("string", 0)

    return {
        "n_rows": n_rows,
        "n_cols": n_cols,
        "group_range": n_groups,
        "n_groups": n_groups,
        "max_group_size": max_group_size,
        "int_count": int_count,
        "float_count": float_count,
        "str_count": str_count,
        "agg_function": _AGG_MAPPING.get(agg_function, 1),
    }

In [ ]:
%%time
### cell 26 ###

course[course["course_code"].str.startswith("CS")].value_counts()

In [ ]:
%%time
### cell 27 ###

course

In [ ]:
%%time
### cell 28 ###

course.sort_values("course_rating_mean", ascending=False, inplace=True)

In [ ]:
%%time
### cell 29 ###

course.reset_index(inplace=True)

In [ ]:
%%time
### cell 30 ###

course.set_index("course_unit", inplace=True)

In [ ]:
%%time
### cell 31 ###

course.loc["KOREA", "course_rating_mean"].value_counts()

In [ ]:
%%time
### cell 32 ###

KOREA = course.loc["KOREA", :]

In [ ]:
%%time
### cell 33 ###

course.loc["CHINA", "course_rating_mean"].value_counts()

In [ ]:
%%time
### cell 34 ###

china = course.loc["CHINA", :]

In [ ]:
%%time
### cell 35 ###

course.loc["CHINA", "course_rating_mean"].value_counts()

In [ ]:
%%time
### cell 36 ###

span = course.loc["SPAN", :]

In [ ]:
%%time
### cell 37 ###

course.loc["CS", "course_rating_mean"].value_counts()

In [ ]:
%%time
### cell 38 ###

cs = course.loc["CS", :]
cs

In [ ]:
%%time
### cell 39 ###

cs_mean = (
    cs.groupby("course_code")
    .mean(numeric_only=True)
    .sort_values("course_rating_int", ascending=False)
)
cs_mean

In [ ]:
%%time
### cell 40 ###

course.loc["WKRPT", "course_rating_mean"].value_counts()

In [ ]:
%%time
### cell 41 ###

wkrpt = course.loc["WKRPT", :]
wkrpt

In [ ]:
%%time
### cell 42 ###

# Skip the default groupkey sorting to save time, then sort only by the desired column
wkrpt_mean = (
    wkrpt.groupby("course_code", sort=False)
    .mean(numeric_only=True)
    .sort_values("course_rating_int", ascending=False)
)
wkrpt_mean

In [ ]:
%%time
### cell 43 ###

wkrpt.groupby("course_code").mean(numeric_only=True)

In [ ]:
%%time
### cell 44 ###

course.loc["PD", "course_rating_mean"].value_counts()

In [ ]:
%%time
### cell 45 ###

pd_df = course.loc["PD", :]
pd_df

In [ ]:
%%time
### cell 46 ###

pd_mean = (
    pd_df.groupby("course_code")
    .mean(numeric_only=True)
    .sort_values("course_rating_int", ascending=False)
)
pd_mean